# T01: The Baseline RAG Pipeline

This notebook builds a working RAG pipeline step by step on the PrimePack AG documents, then probes where it fails. Run the cells top to bottom; change queries and parameters freely.

| Notebook | Focus |
|---|---|
| **T01** | Baseline pipeline (this notebook) |
| T02, T03 | Ingestion, chunking, multimodal documents |
| T04, T05 | Measuring answer quality |
| T06 | Structured outputs |
| T07 to T09 | Retrieval, conversation, query techniques |
| T10 to T14 | Tools and agents |

## Why RAG?

**PrimePack AG** buys packaging (pallets, cardboard boxes, tape) from several suppliers. Employees must answer questions such as:

> *What is the GWP of the Logypal 1 pallet, and is the figure verified?*
> *Can we tell a customer that the tesa tape is PFAS-free?*

A plain LLM cannot answer these reliably:

| Problem | Why it matters |
|---|---|
| No access to internal documents | The answer is not in its training data |
| Hallucination | It may invent plausible figures |
| No evidence trail | Answers cannot be traced to a source |

**Retrieval-Augmented Generation (RAG)** first searches the documents, then lets the LLM answer from the retrieved excerpts only:

```
 Documents ──► Chunker ──► Embedder ──► Vector DB
                                              │
 User query ─────────────────► Embedder ─────►  Retriever ──► Top-k Chunks
                                                                      │
                                                               LLM + Prompt
                                                                      │
                                                               Answer + Sources
```

RAG moves the problem from hallucination to **retrieval quality**: if the right chunk is not retrieved, the answer is still wrong. T02 to T09 work on that.

## Core Concepts

- **Chunk**: a short excerpt of a document (a PDF section, a spreadsheet sheet) with `title`, `content`, `mime_type` and `metadata` such as `source_file`. Chunks are the unit of indexing and retrieval.
- **Embedding**: a vector representing the meaning of a text; similar texts get similar vectors. We use OpenAI's `text-embedding-3-small` (1536 dimensions).
- **Vector store**: stores chunk embeddings on disk and returns the `top_k` chunks closest to a query embedding. We use ChromaDB.
- **Retriever**: wraps a vector store behind `retrieve(query)`.
- **RAG agent**: retrieves chunks for a query, passes them to the LLM as `<source>` excerpts and returns the answer with its sources.

## Setup

Requires `OPENAI_API_KEY` (locally: in `.env.local` at the repo root).

In [ ]:
import sys
from collections import Counter
from statistics import median

import numpy as np
from loguru import logger

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from ki_kompass_rag.baseline_rag import (
    EMBEDDING_MODEL,
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    build_vector_store,
    inspect_retrieval,
    load_chunks,
)
from ki_kompass_rag.ingestion import EMBEDDING_TOKEN_LIMIT, estimate_tokens

logger.remove()
logger.add(sys.stderr, level="WARNING")

llm = build_llm("openai")

## The LLM on Its Own

An LLM only knows what was in its training data. First a general question:

In [ ]:
async def ask_llm(question: str) -> str:
    message = LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=question)]
    )
    response = await llm.generate([message])
    return "".join(c.text or "" for c in response.content)


print(await ask_llm("What does GWP stand for, and what unit is it measured in?"))

Now a question about PrimePack AG's portfolio:

In [ ]:
print(
    await ask_llm(
        "What is the GWP of the Logypal 1 pallet sold by PrimePack AG, and is the figure third-party verified? Provide the link to PrimePack AG's website."
    )
)

PrimePack AG is fictional. A specific figure or link is hallucinated; "I don't know" is honest but useless. GPT-4o-mini is trained to decline more often than small local models, which tend to invent an answer. Either way, the verified figure needs the actual EPD.

### OpenAI or a Local Model?

| | OpenAI API (`gpt-4o-mini`) | Local model via Ollama |
|---|---|---|
| Data | Sent to OpenAI | Stays on the machine |
| Quality | Strong instruction following | Depends on the model and its size |
| Cost | Per token | Hardware and electricity |

The workshop uses OpenAI. `build_llm("ollama", model_name)` returns a local model with the same interface.

### Cost Estimation

`gpt-4o-mini` costs $0.15 per 1M input tokens and $0.60 per 1M output tokens ([pricing](https://developers.openai.com/api/docs/pricing/)). One token is about 4 characters of English text. A RAG query sends the system prompt, `top_k` chunks and the question; assume ~400 tokens per chunk and a 150-token answer:

In [ ]:
INPUT_PRICE = 0.15 / 1_000_000  # USD per token
OUTPUT_PRICE = 0.60 / 1_000_000

tokens_per_chunk = 400
question = "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?"
input_tokens = (
    estimate_tokens(SYSTEM_PROMPT)
    + RETRIEVER_TOP_K * tokens_per_chunk
    + estimate_tokens(question)
)
output_tokens = 150
cost = input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE

print(f"~{input_tokens:,} input + {output_tokens} output tokens: ${cost:.5f} per query")
for n in (1_000, 100_000):
    print(f"{n:>9,} queries/day: ${cost * n:,.2f}")

Input tokens dominate, so cost grows with `top_k` and chunk size.

## Step 1: Load and Chunk Documents

`load_chunks()` converts every file in `data/` to Markdown and splits it at headings (PDFs with docling, Excel one chunk per sheet). Sections longer than `MAX_CHUNK_CHARS` (4,000 characters) are split again at paragraph breaks. Files over 20 MB are skipped. This takes a few minutes on CPU. T02 compares parsers and chunking strategies.

In [ ]:
chunks = load_chunks()

lengths = [len(c.content) for c in chunks]
files = Counter(c.metadata["source_file"] for c in chunks)
too_long = sum(estimate_tokens(c.content) > EMBEDDING_TOKEN_LIMIT for c in chunks)
print(f"{len(chunks)} chunks from {len(files)} files")
print(f"Median length: {median(lengths):.0f} chars, longest: {max(lengths):,} chars")
print(f"Over the {EMBEDDING_TOKEN_LIMIT}-token embedding limit: {too_long}")

In [ ]:
for c in chunks[:3]:
    print(f"[{c.metadata['source_file']}] {c.title!r} ({len(c.content)} chars)")
    print(f"{c.content[:200].strip()!r}\n")

## Step 2: Embed Chunks and Build the Vector Store

Each chunk's `content` is embedded and stored in a ChromaDB collection at `backend/db/data_vs.db`. `reset=True` rebuilds the store from scratch so it always matches the current chunks; T02 to T07 reuse it without re-embedding.

| | `text-embedding-3-small` (used here) | Sentence-transformers models, e.g. `all-MiniLM-L6-v2` |
|---|---|---|
| Dimensions | 1536 | 384 to 1024 |
| Context limit | 8191 tokens | 256 to 8192 tokens, model-dependent |
| Cost | ~$0.02 per 1M tokens | Free, runs locally (slow on CPU) |
| Data | Sent to OpenAI | Stays local |

Any model on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) that works with `sentence-transformers` can be used via `SentenceTransformerEmbeddings(model_name)`.

ChromaDB runs embedded (no server) and suits up to ~100k chunks with a single writer. The toolkit also has `PGVectorStore` (PostgreSQL + pgvector) for concurrent access and SQL filtering.

In [ ]:
embedding_model = OpenAIEmbeddings(EMBEDDING_MODEL)
vector_store = await build_vector_store(chunks, embedding_model, reset=True)
print(f"{vector_store.collection.count()} chunks in the store")

### Similarity in Embedding Space

Cosine similarity measures how close two embeddings point: 1 means identical direction, around 0 unrelated. Change the sentences and compare. ([Embedding projector](https://projector.tensorflow.org) for a visual impression.)

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


anchor = "carbon footprint of a pallet"
others = [
    "GWP value for the Logypal 1",
    "PFAS-free tape declaration",
    "the annual report of a software firm",
]
vectors = await embedding_model.get_embeddings([anchor, *others])
for text, vector in zip(others, vectors[1:], strict=True):
    print(f"{cosine(vectors[0], vector):.3f}  {anchor!r} vs {text!r}")

### Comparing Embedding Models

A quick check: how much more similar is a relevant chunk to the query than an irrelevant one? A larger gap means the model separates useful from noisy chunks better. `all-MiniLM-L6-v2` is downloaded on first use (~90 MB). One example is not an evaluation; T04 measures retrieval properly.

In [ ]:
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)

query = "What is the carbon footprint of the Logypal 1 pallet?"
relevant = "Logypal 1 internal LCA (2020, non-verified): GWP 4.1 kg CO2e per pallet over a 50-trip lifetime."
irrelevant = "All suppliers must comply with applicable environmental regulations and report annually on progress."
models = {
    "all-MiniLM-L6-v2 (local, 384-dim)": SentenceTransformerEmbeddings(
        "sentence-transformers/all-MiniLM-L6-v2"
    ),
    "text-embedding-3-small (API, 1536-dim)": embedding_model,
}

print(f"{'model':<40}{'relevant':>10}{'irrelevant':>12}{'gap':>8}")
for name, model in models.items():
    q, rel, irr = await model.get_embeddings([query, relevant, irrelevant])
    print(
        f"{name:<40}{cosine(q, rel):>10.3f}{cosine(q, irr):>12.3f}{cosine(q, rel) - cosine(q, irr):>8.3f}"
    )

## Step 3: Inspect Retrieval

If the retrieved chunks are wrong, the answer will be wrong no matter how good the LLM is. Check retrieval before blaming the model.

`inspect_retrieval()` prints the top-k chunks with their `score`, the cosine similarity between query and chunk (higher is more similar).

In [ ]:
results = await inspect_retrieval(
    "What materials is the Logypal 1 pallet made from?",
    vector_store,
    embedding_model,
    top_k=RETRIEVER_TOP_K,
)

### A Product Outside the Portfolio

No document mentions a "Lara pallet". Compare the scores with the query above.

In [ ]:
results = await inspect_retrieval(
    "What materials is the Lara pallet made from?",
    vector_store,
    embedding_model,
    top_k=RETRIEVER_TOP_K,
)

The retriever has no notion of "no match": it always returns the closest chunks. Scores are lower, but the LLM still receives chunks about other pallets.

## Step 4: Build the RAG Agent

`build_agent()` combines a `VectorStoreRetriever` (vector store + embedding model), the LLM and a system prompt. We start with a deliberately minimal prompt; the probes below show what it misses.

In [ ]:
BASIC_PROMPT = (
    "You are a helpful AI assistant specialised in sustainability and product compliance for PrimePack AG.\n\n"
    "You will receive document excerpts relevant to the user's question. Produce the best possible answer using only the information in those excerpts."
)

agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=BASIC_PROMPT,
    number_query_expansion=0,  # see T09
)

## Step 5: Ask a Question

`ask()` retrieves the top-k chunks, sends them with the question to the LLM, and prints the answer and the files of the retrieved chunks.

In [ ]:
answer = await ask(agent, "What materials is the Logypal 1 pallet made from?")

## Probing Failure Modes

The documents contain four deliberate traps.

### a) Product Outside the Portfolio

A good answer says the Lara pallet is not in any document instead of describing another pallet.

In [ ]:
answer = await ask(agent, "What materials is the Lara pallet made from?")

### b) Missing Data

The LogyLight datasheet lists all LCA figures as "not yet available". The correct answer says so, without inventing a figure.

In [ ]:
answer = await ask(agent, "What is the GWP of the LogyLight pallet?")

### c) Conflicting Sources

The June 2021 Relicyc datasheet reports 4.1 kg CO₂e per pallet from an internal, non-verified LCA. The 2023 EPD (S-P-09135) is third-party verified. The answer should flag the conflict and prefer the EPD.

In [ ]:
answer = await ask(
    agent, "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?"
)

### d) Unverified Supplier Claim

The tesa brochure claims a 68% CO₂ reduction from an internal assessment, without an EPD. The answer should report the claim as unverified.

In [ ]:
answer = await ask(
    agent,
    "How much lower is the carbon footprint of tesa ECO tape compared to standard tape?",
)

### A Stricter System Prompt

`SYSTEM_PROMPT` in `baseline_rag.py` adds rules for exactly these cases. It is used from T04 on. Rerun the probes with it:

In [ ]:
print(SYSTEM_PROMPT)

strict_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)
for query in [
    "What materials is the Lara pallet made from?",
    "What is the GWP of the LogyLight pallet?",
    "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?",
    "How much lower is the carbon footprint of tesa ECO tape compared to standard tape?",
]:
    print(f"\n>>> {query}")
    answer = await ask(strict_agent, query)

## Summary

| Step | Function |
|---|---|
| 1. Load and chunk | `load_chunks()` |
| 2. Embed and index | `build_vector_store(chunks, embedding_model, reset=True)` |
| 3. Inspect retrieval | `inspect_retrieval(query, vector_store, embedding_model, top_k)` |
| 4. Build agent | `build_agent(vector_store, embedding_model, llm, top_k, system_prompt)` |
| 5. Answer | `ask(agent, query)` |

`run_pipeline("openai", query=...)` runs steps 2 to 5 on the existing store in one call.